Control

In [0]:
from pyspark.sql import functions as F

tabla_control = "workspace.default.control_cargas_nyctaxi"
tabla_bronze = "workspace.default.bronze_nyctaxi"
tabla_silver = "workspace.default.silver_nyctaxi"
tabla_quarantine = "workspace.default.quarantine_nyctaxi"
tabla_gold_diario = "workspace.default.gold_nyctaxi_diario"
tabla_gold_mensual = "workspace.default.gold_nyctaxi_mensual"

print("Tablas de control configuradas correctamente.")

Conteo comparativo entre etapas

In [0]:
conteo_bronze = spark.table(tabla_bronze).count()
conteo_silver = spark.table(tabla_silver).count()
conteo_quarantine = spark.table(tabla_quarantine).count()

conteo_nuevos = (
    spark.table(tabla_control)
    .agg(F.sum("_registros_nuevos").alias("total"))
    .collect()[0]["total"]
)

conteo_duplicados = (
    spark.table(tabla_control)
    .agg(F.sum("_registros_duplicados").alias("total"))
    .collect()[0]["total"]
)

conteo_rechazados = (
    spark.table(tabla_control)
    .agg(F.sum("_registros_rechazados").alias("total"))
    .collect()[0]["total"]
)

print(f"Bronze:             {conteo_bronze}")
print(f"Silver:             {conteo_silver}")
print(f"Quarantine:         {conteo_quarantine}")
print(f"Únicos aceptados:   {conteo_nuevos}")
print(f"Duplicados:         {conteo_duplicados}")
print(f"Rechazados:         {conteo_rechazados}")

Consolidacion por cargas

In [0]:
df_control_cargas = (
    spark.table(tabla_control)
    .withColumn(
        "_registros_clasificados",
        F.col("_registros_nuevos")
        + F.col("_registros_duplicados")
        + F.col("_registros_rechazados")
    )
    .withColumn(
        "_conciliacion",
        F.when(
            F.col("_registros_leidos")
            == F.col("_registros_clasificados"),
            "OK"
        ).otherwise("ERROR")
    )
    .select(
        "_archivo_origen",
        "_registros_leidos",
        "_registros_nuevos",
        "_registros_duplicados",
        "_registros_rechazados",
        "_registros_clasificados",
        "_conciliacion"
    )
    .orderBy("_archivo_origen")
)

display(df_control_cargas)

Consolidadcion global

In [0]:
total_leidos = (
    spark.table(tabla_control)
    .agg(F.sum("_registros_leidos").alias("total"))
    .collect()[0]["total"]
)

total_nuevos = (
    spark.table(tabla_control)
    .agg(F.sum("_registros_nuevos").alias("total"))
    .collect()[0]["total"]
)

total_duplicados = (
    spark.table(tabla_control)
    .agg(F.sum("_registros_duplicados").alias("total"))
    .collect()[0]["total"]
)

total_rechazados = (
    spark.table(tabla_control)
    .agg(F.sum("_registros_rechazados").alias("total"))
    .collect()[0]["total"]
)

total_clasificado = (
    total_nuevos
    + total_duplicados
    + total_rechazados
)

conciliacion_ok = (
    total_leidos == total_clasificado
    and conteo_bronze == total_leidos
    and conteo_silver == total_nuevos
    and conteo_quarantine == total_rechazados
)

print(f"Total registros leídos:       {total_leidos}")
print(f"Total registros nuevos:       {total_nuevos}")
print(f"Total registros duplicados:   {total_duplicados}")
print(f"Total registros rechazados:   {total_rechazados}")
print(f"Total registros clasificados: {total_clasificado}")
print()
print(f"Conciliación general: {'OK' if conciliacion_ok else 'ERROR'}")

Consolidacion de GOLD

In [0]:
conteo_gold_diario = (
    spark.table(tabla_gold_diario)
    .agg(F.sum("cantidad_viajes").alias("total"))
    .collect()[0]["total"]
)

conteo_gold_mensual = (
    spark.table(tabla_gold_mensual)
    .agg(F.sum("cantidad_viajes").alias("total"))
    .collect()[0]["total"]
)

gold_diario_ok = conteo_gold_diario == conteo_silver
gold_mensual_ok = conteo_gold_mensual == conteo_silver

print(f"Silver:                  {conteo_silver}")
print(f"Gold diario - viajes:    {conteo_gold_diario}")
print(f"Gold mensual - viajes:   {conteo_gold_mensual}")
print()
print(
    f"Control Gold diario: "
    f"{'OK' if gold_diario_ok else 'ERROR'}"
)
print(
    f"Control Gold mensual: "
    f"{'OK' if gold_mensual_ok else 'ERROR'}"
)

Estados de las cargas

In [0]:
cargas_error = (
    spark.table(tabla_control)
    .filter(F.col("_estado") != "OK")
    .count()
)

total_cargas = (
    spark.table(tabla_control)
    .count()
)

control_cargas_ok = cargas_error == 0

print(f"Total de cargas:       {total_cargas}")
print(f"Cargas con error:      {cargas_error}")
print()
print(
    f"Estado de las cargas: "
    f"{'OK' if control_cargas_ok else 'ERROR'}"
)